In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms

import medicalDataLoader_Enes as medicalDataLoader
from progressBar import printProgressBar
from utils_Enes import (to_var, getTargetSegmentation, validate, show_predictions, show_augmentations,
                        DiceLoss, show_confusion)
from UNet_Base_Enes import UNet

In [ ]:
# Réglages par défaut (le meilleur choix trouvé jusqu'ici)
MODEL_NAME = 'final_dice'      # dossier où le meilleur modèle est enregistré : models/<MODEL_NAME>/
EPOCHS = 100            # 100 plutôt que 50 : meilleur sur les 10 dernières époques (essai du 16/11)
BATCH_SIZE = 8
LR = 0.001
AUGMENT = True          # augmentation de données (séance 5) : rotation + les options ci-dessous
AUG_FLIP = False        # retournements : retirés, ils mettent le VD du mauvais côté (essai du 16/11)
AUG_SCALE = True        # zoom de 0.9 à 1.1 et décalage
AUG_INTENSITY = True    # luminosité et contraste
SCHEDULER = True        # le lr baisse au fil des époques (cosinus), séance 4
WEIGHT_DECAY = 0        # régularisation L2 (séance 5) : essayé à 1e-4, gain dans le bruit, pas gardé
LOSS = 'ce+dice'        # 'ce' = entropie croisée ; 'ce+dice' : +0,03 de Dice, surtout le myocarde (essai du 16/11)
SEED = 0

# Expériences à lancer à la suite : chacune ne donne que ce qui change par rapport aux réglages ci-dessus.
# [] = un seul entraînement avec les réglages par défaut, puis les figures.
# Déjà fait :
#   recherche du lr ({'name': 'lr_0.0003', 'LR': 3e-4}, ...) -> on garde LR = 0.001
#   augmentation avec / sans retournements, scheduler avec retournements -> on retire les retournements
#   scheduler, scheduler + weight decay, scheduler + 100 époques -> scheduler + 100 époques
#   (« sans_flip_scheduler_100ep » = la graine 0 avec l'entropie croisée)
#   loss CE seule ou CE + Dice -> CE + Dice (0,817 contre 0,786 sur les 10 dernières époques)
# Plus tard, si on a le temps : refaire avec les graines 1 et 2 ('SEED': 1) pour une moyenne ± écart-type
EXPERIMENTS = []

# Dossier des données : Kaggle si on y est, sinon le dossier Data à côté du dépôt
if os.path.exists('/kaggle/input'):
    DATA_DIR = '/kaggle/input/datasets/enesozkosar/acdc-heartbeat/Data/'
else:
    DATA_DIR = '../Data/'

In [ ]:
# Ce que le réseau voit pendant l'entraînement (avec les réglages d'augmentation ci-dessus)
transform = transforms.Compose([transforms.ToTensor()])
preview_set = medicalDataLoader.MedicalImageDataset('train', DATA_DIR, transform=transform, mask_transform=transform,
                                                    augment=AUGMENT, flip=AUG_FLIP, scale=AUG_SCALE,
                                                    intensity=AUG_INTENSITY)
show_augmentations(preview_set, index=100)

In [ ]:
def runTraining():
    # Même graine à chaque lancement : deux expériences partent du même point
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    print('Dataset :', DATA_DIR)
    transform = transforms.Compose([transforms.ToTensor()])

    # train_set sert à apprendre (avec ou sans augmentation) ;
    # train_eval_set contient les mêmes images, jamais modifiées, pour mesurer le Dice d'entraînement
    train_set = medicalDataLoader.MedicalImageDataset('train', DATA_DIR, transform=transform,
                                                      mask_transform=transform, augment=AUGMENT,
                                                      flip=AUG_FLIP, scale=AUG_SCALE, intensity=AUG_INTENSITY)
    train_eval_set = medicalDataLoader.MedicalImageDataset('train', DATA_DIR, transform=transform,
                                                           mask_transform=transform)
    val_set = medicalDataLoader.MedicalImageDataset('val', DATA_DIR, transform=transform,
                                                    mask_transform=transform)
    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
    train_eval_loader = DataLoader(train_eval_set, batch_size=BATCH_SIZE, shuffle=False)
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)

    net = UNet(num_classes=4)
    if torch.cuda.is_available():
        net.cuda()
    print('Paramètres : {:,}'.format(sum(p.numel() for p in net.parameters())))

    CE_loss = nn.CrossEntropyLoss()
    dice_loss = DiceLoss()

    def loss_fn(out, target):
        # L'entropie croisée traite chaque pixel pareil (le fond domine) ;
        # la loss Dice donne le même poids à chaque structure, même petite (VD, myocarde)
        if LOSS == 'ce+dice':
            return CE_loss(out, target) + dice_loss(out, target)
        return CE_loss(out, target)
    # weight_decay : régularisation L2, ajoute WEIGHT_DECAY * poids au gradient de chaque poids
    optimizer = torch.optim.Adam(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    # Scheduler (séance 4) : le lr part de LR et descend en douceur jusqu'à presque 0 à la dernière époque.
    # Grands pas au début pour apprendre vite, petits pas à la fin pour se stabiliser.
    scheduler = None
    if SCHEDULER:
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    model_dir = os.path.join('models', MODEL_NAME)
    os.makedirs(model_dir, exist_ok=True)
    best_path = os.path.join(model_dir, 'best.pth')
    history = {'train_loss': [], 'val_loss': [], 'train_dice': [], 'val_dice': [], 'val_hd': []}
    best_dsc, best_epoch = -1, 0

    for epoch in range(EPOCHS):
        # Entraînement
        net.train()
        losses = []
        for j, (images, labels, _) in enumerate(train_loader):
            images = to_var(images)
            target = getTargetSegmentation(to_var(labels))

            optimizer.zero_grad()
            loss = loss_fn(net(images), target)
            loss.backward()
            optimizer.step()

            losses.append(loss.item())
            printProgressBar(j + 1, len(train_loader), prefix='[Training] Epoch: {} '.format(epoch),
                             length=15, suffix=' Loss: {:.4f}'.format(loss.item()))
        train_loss = np.mean(losses)
        current_lr = optimizer.param_groups[0]['lr']
        if scheduler is not None:
            scheduler.step()

        # Mesures : Dice sur le train (sans Hausdorff, trop lent), Dice et Hausdorff sur la validation
        _, train_dsc, _ = validate(net, train_eval_loader, loss_fn, with_hausdorff=False)
        val_loss, val_dsc, val_hd = validate(net, val_loader, loss_fn)
        mean_dsc = np.mean(val_dsc)
        print('Epoch {:2d} | lr {:.6f} | loss train {:.4f}  val {:.4f}'.format(epoch, current_lr, train_loss, val_loss))
        print('   Train Dice  VD {:.3f}  Myo {:.3f}  VG {:.3f} | moyenne {:.3f}'.format(*train_dsc, np.mean(train_dsc)))
        print('   Val   Dice  VD {:.3f}  Myo {:.3f}  VG {:.3f} | moyenne {:.3f}'.format(*val_dsc, mean_dsc))
        print('   Val   Hausdorff (px)  VD {:.1f}  Myo {:.1f}  VG {:.1f}'.format(*val_hd))

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_dice'].append(train_dsc)
        history['val_dice'].append(val_dsc)
        history['val_hd'].append(val_hd)

        # On garde seulement le meilleur modèle sur la validation
        if mean_dsc > best_dsc:
            best_dsc, best_epoch = mean_dsc, epoch
            torch.save(net.state_dict(), best_path)
            print('   -> meilleur modèle enregistré')

    np.save(os.path.join(model_dir, 'history.npy'), history)
    print('Meilleur Dice moyen : {:.4f} (époque {})'.format(best_dsc, best_epoch))
    last10 = np.mean([np.mean(d) for d in history['val_dice'][-10:]])
    print('Dice moyen des 10 dernières époques : {:.4f}'.format(last10))

    net.load_state_dict(torch.load(best_path))   # le meilleur, pas le dernier
    return net, train_eval_loader, val_loader

In [ ]:
if EXPERIMENTS:
    DEFAULTS = {'MODEL_NAME': MODEL_NAME, 'EPOCHS': EPOCHS, 'BATCH_SIZE': BATCH_SIZE, 'LR': LR,
                'AUGMENT': AUGMENT, 'AUG_FLIP': AUG_FLIP, 'AUG_SCALE': AUG_SCALE,
                'AUG_INTENSITY': AUG_INTENSITY, 'SCHEDULER': SCHEDULER, 'WEIGHT_DECAY': WEIGHT_DECAY, 'LOSS': LOSS, 'SEED': SEED}

    for exp in EXPERIMENTS:
        # On repart des réglages par défaut, puis on applique ce qui change dans cette expérience
        globals().update(DEFAULTS)
        for key, value in exp.items():
            if key != 'name':
                globals()[key] = value
        MODEL_NAME = exp['name']
        print('\n========== {} =========='.format(MODEL_NAME))
        net, train_loader, val_loader = runTraining()
        show_predictions(net, val_loader, title='Validation : ' + MODEL_NAME)
        show_confusion(net, val_loader, title='(' + MODEL_NAME + ')')
    globals().update(DEFAULTS)

    print('\nRésumé des expériences')
    for exp in EXPERIMENTS:
        h = np.load(os.path.join('models', exp['name'], 'history.npy'), allow_pickle=True).item()
        means = [np.mean(d) for d in h['val_dice']]
        print('{:<28} | meilleur {:.4f} (époque {}) | 10 dernières époques {:.4f}'.format(
            exp['name'], max(means), int(np.argmax(means)), np.mean(means[-10:])))
else:
    net, train_loader, val_loader = runTraining()

    # Post-traitement : le même modèle, mesuré sans puis avec (on garde la plus grosse tache par classe)
    CE_loss = nn.CrossEntropyLoss()
    for post in [False, True]:
        _, dsc, hd = validate(net, val_loader, CE_loss, post_process=post)
        print('Validation {} post-traitement'.format('avec' if post else 'sans'))
        print('   Dice  VD {:.3f}  Myo {:.3f}  VG {:.3f} | moyenne {:.3f}'.format(*dsc, np.mean(dsc)))
        print('   Hausdorff (px)  VD {:.1f}  Myo {:.1f}  VG {:.1f}'.format(*hd))

    show_predictions(net, train_loader, title='Entraînement')
    show_predictions(net, val_loader, title='Validation')
    show_predictions(net, val_loader, title='Validation avec post-traitement', post_process=True)
    show_confusion(net, val_loader, title='(sans post-traitement)')
    show_confusion(net, val_loader, title='(avec post-traitement)', post_process=True)